In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS raw.dbt_billingshield

In [0]:
%sql
CREATE OR REPLACE VIEW raw.dbt_billingshield.stg_cms_claims AS
SELECT
    provider_id,
    provider_last_name,
    provider_first_name,
    specialty,
    state,
    city,
    zip_code,
    entity_code,
    procedure_code,
    procedure_desc,
    drug_indicator,
    place_of_service,
    total_beneficiaries,
    total_services,
    total_beneficiary_days,
    avg_submitted_charge,
    avg_medicare_allowed,
    avg_medicare_paid,
    avg_medicare_standardized,
    charge_to_payment_ratio,
    billing_intensity,
    specialty_avg_charge,
    specialty_stddev_charge,
    provider_charge_zscore,
    facility_flag,
    silver_processed_at,
    source_file
FROM raw.silver.cms_claims_features

In [0]:
%sql
CREATE OR REPLACE VIEW raw.dbt_billingshield.int_provider_stats AS
SELECT
    provider_id,
    specialty,
    state,
    city,
    zip_code,
    entity_code,
    COUNT(*)                                                AS total_procedures,
    SUM(total_beneficiaries)                                AS total_beneficiaries,
    SUM(total_services)                                     AS total_services,
    ROUND(SUM(avg_submitted_charge * total_services), 2)    AS total_billed,
    ROUND(SUM(avg_medicare_paid * total_services), 2)       AS total_paid,
    ROUND(AVG(charge_to_payment_ratio), 4)                  AS avg_charge_ratio,
    ROUND(AVG(billing_intensity), 4)                        AS avg_billing_intensity,
    ROUND(MAX(provider_charge_zscore), 4)                   AS max_zscore,
    ROUND(AVG(provider_charge_zscore), 4)                   AS avg_zscore,
    COUNT(DISTINCT procedure_code)                          AS unique_procedures,
    ROUND(AVG(facility_flag), 4)                            AS facility_rate
FROM raw.silver.cms_claims_features
GROUP BY
    provider_id, specialty, state, city, zip_code, entity_code

In [0]:
%sql
CREATE OR REPLACE TABLE raw.dbt_billingshield.fct_provider_risk AS
SELECT
    provider_id,
    specialty,
    state,
    city,
    zip_code,
    entity_code,
    COUNT(*)                                                AS total_procedures,
    SUM(total_beneficiaries)                                AS total_beneficiaries,
    SUM(total_services)                                     AS total_services,
    ROUND(SUM(avg_submitted_charge * total_services), 2)    AS total_billed,
    ROUND(SUM(avg_medicare_paid * total_services), 2)       AS total_paid,
    ROUND(AVG(charge_to_payment_ratio), 4)                  AS avg_charge_ratio,
    ROUND(AVG(billing_intensity), 4)                        AS avg_billing_intensity,
    ROUND(MAX(provider_charge_zscore), 4)                   AS max_zscore,
    ROUND(AVG(provider_charge_zscore), 4)                   AS avg_zscore,
    COUNT(DISTINCT procedure_code)                          AS unique_procedures,
    ROUND(AVG(facility_flag), 4)                            AS facility_rate,
    RANK() OVER (
        PARTITION BY specialty
        ORDER BY MAX(provider_charge_zscore) DESC
    )                                                       AS specialty_rank,
    CASE
        WHEN MAX(provider_charge_zscore) >= 3 THEN 'Critical'
        WHEN MAX(provider_charge_zscore) >= 1 THEN 'Elevated'
        ELSE 'Normal'
    END                                                     AS risk_tier,
    CURRENT_TIMESTAMP()                                     AS dbt_processed_at
FROM raw.silver.cms_claims_features
GROUP BY
    provider_id, specialty, state, city, zip_code, entity_code

num_affected_rows,num_inserted_rows


In [0]:
%sql
SHOW TABLES IN raw.dbt_billingshield

database,tableName,isTemporary
dbt_billingshield,fct_provider_risk,false
dbt_billingshield,int_provider_stats,false
dbt_billingshield,stg_cms_claims,false


In [0]:
%sql
SELECT COUNT(*) as provider_count,
       COUNT(CASE WHEN risk_tier = 'Critical' THEN 1 END) as critical,
       COUNT(CASE WHEN risk_tier = 'Elevated' THEN 1 END) as elevated,
       COUNT(CASE WHEN risk_tier = 'Normal' THEN 1 END) as normal,
       ROUND(SUM(total_billed), 2) as total_billed
FROM raw.dbt_billingshield.fct_provider_risk

provider_count,critical,elevated,normal,total_billed
236681,17890,51910,166881,7.066046479019E10
